# 04 · Órdenes de fabricación (OF)

Para que el modelo tenga sentido, los tiempos reales tienen que ser completos: solo sirven las OF terminadas y con
todas sus imputaciones. Se aplican tres filtros: el estado de la OF, su fecha de creación y su fecha de
finalización prevista.

- **Entrada:** `data/raw/CPRManufacturingOrder.csv`
- **Salida:** `data/processed/df_OF.parquet` y `results/tables/embudo_04.json`

In [9]:
import sys
from pathlib import Path

# Raíz del repositorio (la carpeta que contiene src/), para poder importar src
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))

import pandas as pd

from src.columnas import COLUMNAS_OF, leer_csv
from src.embudo import guardar_embudo, tabla_embudo
from src.limpieza import eliminar_columnas
from src.rutas import PROCESSED

pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

embudo = {}  # número de OF que quedan tras cada paso

## 1. Carga

Solo se trabaja con las OF de la planta 100 (`CodCompany = 100`).

In [10]:
df_OF = leer_csv("CPRManufacturingOrder.csv", COLUMNAS_OF)
print(f"OF en el archivo: {len(df_OF)}")

df_OF = df_OF[df_OF["CodCompany"] == 100]
embudo["OF iniciales"] = len(df_OF)
print(f"OF de la planta 100: {len(df_OF)}")

OF en el archivo: 18106
OF de la planta 100: 18106


## 2. OF en estado "Creada"

Las OF en estado "Creada" nunca han recibido imputaciones de tiempo, así que se eliminan.
El estado "Creada" tiene dos identificadores:

- ID 100: `7776bcc3-98be-48f1-913a-30bbb40ef0bb`
- ID 200: `7776bcc3-98be-48f1-913a-30bbb40ef0bb200`

In [11]:
ESTADO_CREADA = ["7776bcc3-98be-48f1-913a-30bbb40ef0bb", "7776bcc3-98be-48f1-913a-30bbb40ef0bb200"]

mask_creada = df_OF["IDMOSituation"].isin(ESTADO_CREADA)
print(f"OF en estado Creada (se eliminan): {mask_creada.sum()}")

df_OF = df_OF[~mask_creada]
embudo["OF sin estado Creada"] = len(df_OF)
print(f"OF restantes: {len(df_OF)}")

OF en estado Creada (se eliminan): 1
OF restantes: 18105


## 3. OF creadas antes de junio de 2025

Las OF creadas antes del 01/06/2025 se descartan: el sistema estaba en fase de implantación y esos datos no son
fiables.

In [12]:
df_OF["CreationTimestamp"] = pd.to_datetime(df_OF["CreationTimestamp"])

antes = len(df_OF)
df_OF = df_OF[df_OF["CreationTimestamp"] >= "2025-06-01"]
embudo["OF creadas desde junio de 2025"] = len(df_OF)
print(f"OF creadas antes de junio de 2025 (se eliminan): {antes - len(df_OF)}")
print(f"OF restantes: {len(df_OF)}")

OF creadas antes de junio de 2025 (se eliminan): 5303
OF restantes: 12802


## 4. OF que terminan en agosto de 2026 o después

Con el equipo técnico se ha determinado que, a partir de agosto, no se puede garantizar la finalización de todas las
fases de una OF. Por eso se descartan las OF con fecha de finalización prevista (`ManualEndDate`) igual o posterior
al 01/08/2026, y también las que no tienen fecha de finalización.

In [13]:
df_OF["ManualEndDate"] = pd.to_datetime(df_OF["ManualEndDate"], errors="coerce")

antes = len(df_OF)
df_OF = df_OF[df_OF["ManualEndDate"] < "2026-08-01"]  # las fechas vacías (NaT) tampoco cumplen la condición
embudo["OF en intervalo temporal"] = len(df_OF)
print(f"OF que terminan en agosto o después, o sin fecha (se eliminan): {antes - len(df_OF)}")
print(f"OF restantes: {len(df_OF)}")

OF que terminan en agosto o después, o sin fecha (se eliminan): 819
OF restantes: 11983


## 5. Limpieza de atributos

Se prescinde de los atributos innecesarios.

In [14]:
COLUMNAS_SOBRAN = [
    "IDStructure", "TimeUnitDefault", "Notes", "IDWareHouse", "Priority",
    "Color", "Group", "InternalCode", "IDRoute", "AcceptedDate", "CodCompany",
    "IDProject", "IDProTask", "IDProMaterial", "AutomaticCodification",
    "Historic", "RowTimestamp", "PlanningDirection", "LastModifiedBy",
    "IDMaintenanceOrder", "IDOrderMaterialPrevision", "IDLaunchedMRP",
    "AutoBatchAssignSalesOrders", "RowCreatedBy", "CreationTimestamp",
    "IDArticleVersion", "IDSite", "PlannedEndDate", "PlannedStartDate",
    "PlannedEndMRPDate", "ManualPStartDate", "ManualPEndDate", "ManualEndDate",
]

df_OF = eliminar_columnas(df_OF, COLUMNAS_SOBRAN)
df_OF.head(3)

Eliminadas 33 columnas. Quedan 10 columnas y 11983 filas.


,IDManufacturingOrder,CodManufacturingOrder,IDArticle,Description,IDStructureType,Quantity,IDMOSituation,RealEndDate,QuantityBaseFormula,IDUnitBaseFormula
1,0008c8eb-104f-4631-8a7c-50b6d1166cc4,OF2509130,6b526dff-7ba3-4251-99fb-96ff8a71d6eb,Elektrodenschaft F704510502739,d3bd7234-a0af-4c35-b658-dcd8ac943c67,3.0,f57597bd-79b3-4a87-9f62-fdd434de7804,2025-09-26 00:00:00.000,3.0,153b2e67-a636-4e4f-9fa5-fc100da76dfb
2,000be0da-5295-42a4-87cf-bf7036f5df2d,OF2602948,86a77194-ec9d-4c80-aeea-4631abab94d6,Mangon 6028003,d3bd7234-a0af-4c35-b658-dcd8ac943c67,2.0,f57597bd-79b3-4a87-9f62-fdd434de7804,2026-05-22 00:00:00.000,2.0,153b2e67-a636-4e4f-9fa5-fc100da76dfb
3,00111737-d13f-4c82-86c2-995751c98724,OF2600425,bb09abb1-6458-4db9-8dfb-18031995ffb0,BRAS 250610101,d3bd7234-a0af-4c35-b658-dcd8ac943c67,1.0,f57597bd-79b3-4a87-9f62-fdd434de7804,2026-01-29 00:00:00.000,1.0,153b2e67-a636-4e4f-9fa5-fc100da76dfb


## 6. Guardado

In [15]:
df_OF.to_parquet(PROCESSED / "df_OF.parquet")
guardar_embudo(embudo, "04")

## Resumen

In [16]:
tabla_embudo(embudo, "Número de OF")

,Número de OF
OF iniciales,18106
OF sin estado Creada,18105
OF creadas desde junio de 2025,12802
OF en intervalo temporal,11983
